# Example 11.5 / 3: Rotated Cones

Companion notebook for **11.5 Second-Order Cones** in the AMPL lab Documentation, section "Rotated Cones". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Rotated Cones

A rotated SOC has the form

$$2uv\ge\lVert x\rVert_2^2,\qquad u\ge0,\ v\ge0.$$

The nonnegativity conditions are essential. Rotated cones model quadratic-over-linear relations and perspective formulations. Omitting either sign restriction changes the feasible set and destroys the conic guarantee.

Equivalent rotated-cone conventions sometimes omit the factor 2 by rescaling one coordinate.
State the convention before translating a formulation; copying coefficients without the
corresponding rescaling changes the feasible set.

For $x=2$: the point $u=1,\ v=2$ satisfies $2uv=4\ge x^2=4$. The sign-free pair $u=-1,\ v=-2$
also gives $2uv=4$, but it is outside the cone, which is why $u,v\ge0$ must be stated. Under the
convention without the factor 2, $uv\ge x^2$, the same $x$ needs for example $u=1,\ v=4$.

A rotated cone written as `2*u*v >= x^2` with nonnegative `u` and `v` is recognized by Gurobi.
Minimizing $u+v$ with $x=2$ gives $u=v=\sqrt2$ and $u+v=2\sqrt2\approx2.828427$:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
param x = 2;  # fixed coordinate

var u >= 0;  # nonnegativity is part of the cone
var v >= 0;  # nonnegativity is part of the cone

minimize Sum: u + v;  # optimum u = v = sqrt(2)

subject to Rotated: 2 * u * v >= x^2;  # rotated cone 2uv >= x^2

Writing model.mod


### amplpy: run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# run.py
ampl.solve(solver="gurobi", verbose=False)
print(ampl.solve_result, round(ampl.var["u"].value(), 6),
      round(ampl.var["v"].value(), 6), round(ampl.obj["Sum"].value(), 6))

solved 1.414214 1.414214 2.828427
